# Agente RAG - Challenge ALURA ORACLE OCI
Fluxo:

**5 PDFs → carregamento → chunking → embeddings → vector store → 1 ferramenta de busca → agente → resposta**

## 1. Instalação

In [1]:
%pip install -qU pypdf langchain langchain-community langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração

As chaves ficam em variáveis de ambiente.

In [2]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Defina a variável de ambiente GROQ_API_KEY antes de executar o notebook."
    )

## 3. Modelo de linguagem

O LLM será usado somente para interpretar a pergunta, decidir quando usar a ferramenta e formular a resposta final.

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

c:\Users\fabia\Desktop\challenger IA\OracleOne-Challenger_AI_Tech\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Carregando os 5 PDFs

Todos os documentos ficam em `documentos/` e são colocados na **mesma coleção**.
O retriever encontrará os trechos semanticamente mais próximos da pergunta.

In [4]:
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

DOCUMENTOS_DIR = Path("documentos")

pdfs = sorted(DOCUMENTOS_DIR.glob("*.pdf"))

if len(pdfs) != 5:
    raise ValueError(
        f"Esperados 5 PDFs em '{DOCUMENTOS_DIR}', mas foram encontrados {len(pdfs)}."
    )

for pdf in pdfs:
    print(pdf.name)

loader = DirectoryLoader(
    str(DOCUMENTOS_DIR),
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

pages = loader.load()

print(f"Total de páginas/documentos carregados: {len(pages)}")

C:\Users\fabia\AppData\Local\Temp\ipykernel_12260\3488623595.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


Arquitetura_de_Microsservicos_Mapa_de_Dominios.pdf
Guia_Oficial_de_Engenharia_Backend.pdf
Guia_Oficial_de_Engenharia_Frontend.pdf
Manual_de_Onboarding_para_Desenvolvedores.pdf
Manual_Maestro_de_Resiliencia_Resposta_a_Incidentes.pdf


100%|██████████| 5/5 [00:02<00:00,  2.19it/s]

Total de páginas/documentos carregados: 76


## 5. Dividindo os documentos em chunks

Chunks menores ajudam a devolver somente o trecho relevante ao LLM, reduzindo contexto desnecessário e, consequentemente, tokens.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print(f"Total de chunks: {len(chunks)}")

Total de chunks: 206


## 6. Embeddings

Usamos um modelo multilíngue, adequado aos documentos e perguntas em português.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2565.92it/s]


## 7. Criando uma única base vetorial

A base contém os chunks dos cinco PDFs.

In [7]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    chunks,
    embedding=embed_model
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

print("Vector store criada com sucesso.")

Vector store criada com sucesso.


## 8. Testando a recuperação

Antes de criar o agente, verificamos se a busca encontra trechos relevantes.

In [8]:
query = "Como fazer commits pequenos e descritivos?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):
    print(f"--- Resultado {i} ---")
    print(f"Arquivo: {doc.metadata.get('source')}")
    print(f"Página: {doc.metadata.get('page')}")
    print(doc.page_content[:800])
    print()

--- Resultado 1 ---
Arquivo: documentos\Manual_de_Onboarding_para_Desenvolvedores.pdf
Página: 10
1. Atualize sua branch develop local:
```bash
git checkout develop
git pull origin develop
```
2. Crie sua branch de feature:
```bash
git checkout -b feature/PEG-1345-campo-observacoes-agendamento
```
3. Desenvolva e faça commits pequenos e descritivos (padrão Conventional Commits):
```bash
git add src/main/java/com/pegasus/agendio/model/Agendamento.java
git commit -m "feat(PEG-1345): adiciona campo observacoes na entidade Agendamento"
git add src/main/java/com/pegasus/agendio/dto/AgendamentoDTO.java
git commit -m "feat(PEG-1345): inclui campo observacoes no DTO de resposta"
git add src/test/java/com/pegasus/agendio/AgendamentoServiceTest.java
git commit -m "test(PEG-1345): adiciona testes unitarios para campo observacoes"
```
4. Suba sua branch para o repositório remoto:
```bash

--- Resultado 2 ---
Arquivo: documentos\Manual_de_Onboarding_para_Desenvolvedores.pdf
Página: 10
```
4. Suba su

## 9. Única ferramenta do agente

O agente terá apenas uma ferramenta: buscar contexto nos cinco PDFs.

Além do texto, retornamos o nome do arquivo e a página para facilitar a rastreabilidade da resposta.

In [9]:
from langchain_core.tools import tool

@tool
def pega_contexto(query: str) -> str:
    """Busca nos cinco PDFs os trechos mais relevantes para responder à pergunta do usuário."""

    docs = retriever.invoke(query)

    if not docs:
        return "Nenhum trecho relevante foi encontrado nos documentos."

    resultados = []

    for doc in docs:
        fonte = Path(doc.metadata.get("source", "arquivo desconhecido")).name
        pagina = doc.metadata.get("page")

        resultados.append(
            f"Fonte: {fonte}\n"
            f"Página: {pagina}\n"
            f"Conteúdo:\n{doc.page_content}"
        )

    return "\n\n---\n\n".join(resultados)

## 10. Criando o agente

O agente só precisa consultar a base documental quando necessário.

In [10]:
from langchain.agents import create_agent

system_prompt = """
Você é um assistente de perguntas e respostas sobre os documentos disponibilizados.

Regras:
1. Use a ferramenta de busca para encontrar informações nos documentos antes de responder.
2. Responda somente com informações presentes nos documentos.
3. Não use conhecimento externo para completar lacunas.
4. Se a informação não estiver nos documentos recuperados, responda:
   "Não sei a resposta com base nos documentos."
5. Responda de forma clara e objetiva.
"""

agente_pdf = create_agent(
    model=llm,
    tools=[pega_contexto],
    system_prompt=system_prompt
)

## 11. Função para fazer perguntas

In [11]:
def perguntar(pergunta: str) -> str:
    resultado = agente_pdf.invoke({
        "messages": [
            ("user", pergunta)
        ]
    })

    return resultado["messages"][-1].content

## 12. Testes

Faça perguntas diretamente relacionadas ao conteúdo dos cinco PDFs.

In [12]:
perguntas = [
    "Como fazer commits pequenos e descritivos?",
    "Como funciona a arquitetura de microsserviços?",
    "O que é frontend?",
    "Como devemos lidar com uma falha em produção?",
    "Como funciona o ecossistema tecnológico principal?"
]

for pergunta in perguntas:
    print(f"PERGUNTA: {pergunta}")
    print(f"RESPOSTA: {perguntar(pergunta)}")
    print("\n" + "=" * 80 + "\n")

PERGUNTA: Como fazer commits pequenos e descritivos?
RESPOSTA: Para fazer commits pequenos e descritivos, você deve seguir o padrão Conventional Commits. Isso significa que cada commit deve ter um tipo específico, como "feat" para novas funcionalidades, "fix" para correções de bugs, "chore" para manutenção de build ou dependências, e "docs" para mudanças na documentação. Além disso, a descrição do commit deve ser curta e descritiva, informando o que foi alterado ou adicionado. Por exemplo: "feat(PEG-1345): adiciona campo observacoes na entidade Agendamento". É importante fazer commits pequenos e frequentes, para que seja fácil identificar as alterações feitas no código.


PERGUNTA: Como funciona a arquitetura de microsserviços?
RESPOSTA: A arquitetura de microsserviços funciona da seguinte forma:

*   Cada microsserviço é responsável por um domínio específico de negócio e é projetado para ser escalável de forma independente.
*   Os microsserviços se comunicam entre si por meio de APIs 

## 13. Teste de informação fora dos documentos

Esta pergunta serve para verificar se o agente respeita a regra de não inventar respostas.

In [ ]:
print(perguntar("Como vamos para a Lua?"))

## 14. Estrutura esperada do projeto

```text
projeto/
├── projeto.ipynb
└── documentos/
    ├── Arquitetura_de_Microsservicos_Mapa_de_Dominios.pdf
    ├── Engenharia_Backend.pdf
    ├── Engenharia_Frontend.pdf
    ├── Onboarding_para_Desenvolvedores.pdf
    └── Resiliencia_Resposta_a_Incidentes.pdf
```